# 1 - Mise à jour de la base de données

Ce notebook illustre les fonctionnalités des modules `updater` et `deleter` permettant de mettre à jour, supprimer et gérer explicitement les colonnes d'une base DuckLake. Les connexions sont créées via `DuckLakeConnector` et passées aux classes `DatabaseUpdater` et `DatabaseDeleter`.

Rappel de schéma (§2 de `specification-bdd.md`) : il n'existe **aucune table `dim_*`** — les colonnes catégorielles stockent directement leurs libellés dans `fact_table`, et `is_categorical` est une pure métadonnée d'UI **inférée une seule fois**, à la création de la colonne : les mises à jour et suppressions ne la recalculent jamais ; elle se corrige explicitement par `update_column_metadata(col, is_categorical=...)`.

## 0 - Importation des modules

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import shutil
import sys
from datetime import datetime, timedelta

import narwhals as nw
import numpy as np
import pandas as pd
import polars as pl

# Ajout du chemin vers le package parent
sys.path.append("..")

# Importation des modules ad hoc
from dt_ducklake_manager.connection import DuckLakeConnector
from dt_ducklake_manager.operations.deleter import DatabaseDeleter
from dt_ducklake_manager.operations.updater import DatabaseUpdater
from dt_ducklake_manager.schema import DuckLakeTablesBuilder

# Paramètres globaux
CATEGORICAL_THRESHOLD: int = 10
PRIMARY_KEYS: list = [
    "indicator",
    "country",
    "kind",
    "model",
    "training",
    "week",
    "horizon",
    "date",
]
CATALOG_PATH: str = os.path.join("../outputs", "database_update_demo.ducklake")
DATA_PATH: str = os.path.join("../outputs", "database_update_demo_data/")

## 1 - Construction de la base de données initiale

Construction d'un jeu de données contrôlé (~80 lignes) conçu pour illustrer, plus loin, que le statut catégoriel (`is_categorical`) inféré à la construction reste stable quand les modalités évoluent.

**État initial visé (`CATEGORICAL_THRESHOLD=10`) :**
- `source` : 14 valeurs uniques > seuil → **non-catégorielle**
- `model` : 3 valeurs uniques ≤ seuil → **catégorielle**

In [ ]:
# Initialisation du générateur aléatoire pour la reproductibilité
np.random.seed(0)

# Définition des modalités
indicators_init = ["temperature", "humidity", "pressure", "wind_speed"]
countries_init = ["France", "Germany", "Italy", "Spain", "Belgium"]
kinds_init = ["forecast", "observation"]
models_init = ["model_A", "model_B", "model_C"]
trainings_init = ["train_v1", "train_v2"]
horizons_init = [1, 7, 14, 30]
weeks_init = list(range(1, 5))

# Colonne source : 14 valeurs uniques → non-catégorielle (> CATEGORICAL_THRESHOLD=10)
sources_init = [f"source_{i:02d}" for i in range(1, 16)]

# Labels en français
labels_init: dict = {
    "indicator": "Indicateur",
    "country": "Pays",
    "kind": "Type",
    "model": "Modèle",
    "training": "Entraînement",
    "week": "Semaine",
    "horizon": "Horizon",
    "date": "Date",
    "value": "Valeur",
    "lower_bound": "Borne inférieure",
    "upper_bound": "Borne supérieure",
    "quality_score": "Score de qualité",
    "source": "Source",
    "notes": "Notes",
}

# Génération de 80 lignes (les doublons sur la clé primaire seront supprimés)
start_date_init = datetime(2024, 1, 1)
rows_init = []
for i in range(80):
    date = start_date_init + timedelta(days=i % 15)
    rows_init.append(
        {
            "indicator": np.random.choice(indicators_init),
            "country": np.random.choice(countries_init),
            "kind": np.random.choice(kinds_init),
            "model": np.random.choice(models_init),
            "training": np.random.choice(trainings_init),
            "week": weeks_init[i % len(weeks_init)],
            "horizon": horizons_init[i % len(horizons_init)],
            "date": date,
            "value": np.random.uniform(10, 100),
            "lower_bound": None
            if np.random.random() > 0.7
            else np.random.uniform(5, 50),
            "upper_bound": None
            if np.random.random() > 0.7
            else np.random.uniform(50, 150),
            "quality_score": np.random.uniform(0, 1),
            "source": np.random.choice(sources_init),
            "notes": np.random.choice(["OK", "Warning", None], p=[0.7, 0.2, 0.1]),
        }
    )

# Création et dédoublonnage du DataFrame sur la clé primaire composite
df_demo = pd.DataFrame(rows_init)
df_demo["date"] = pd.to_datetime(df_demo["date"])
df_demo = df_demo.drop_duplicates(subset=PRIMARY_KEYS, keep="first").reset_index(
    drop=True
)

# Vérification des contraintes de cardinalité
assert df_demo["source"].nunique() > CATEGORICAL_THRESHOLD, (
    f"source doit avoir > {CATEGORICAL_THRESHOLD} valeurs uniques "
    f"(actuel : {df_demo['source'].nunique()})"
)
assert df_demo["model"].nunique() <= CATEGORICAL_THRESHOLD, (
    f"model doit avoir ≤ {CATEGORICAL_THRESHOLD} valeurs uniques "
    f"(actuel : {df_demo['model'].nunique()})"
)

# Affichage
print(f"Lignes après dédoublonnage   : {len(df_demo)}")
print(f"source — valeurs uniques     : {df_demo['source'].nunique()} (non-catégoriel)")
print(f"model  — valeurs uniques     : {df_demo['model'].nunique()}  (catégoriel)")
df_demo.head()

In [ ]:
# Suppression du catalogue et des données existants pour garantir un état initial propre
for suffix in ["", ".wal"]:
    path_to_remove = CATALOG_PATH + suffix
    if os.path.exists(path_to_remove):
        os.remove(path_to_remove)
        print(f"Fichier supprimé : {path_to_remove}")
if os.path.exists(DATA_PATH):
    shutil.rmtree(DATA_PATH)
    print(f"Répertoire supprimé : {DATA_PATH}")

# Création de la connexion DuckLake et construction du schéma initial
conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()
builder = DuckLakeTablesBuilder(
    df=df_demo,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=PRIMARY_KEYS,
    connection=conn,
)
builder.build_schema(column_labels=labels_init)
conn.close()

print(f"\nBase de données créée : {CATALOG_PATH}")

In [ ]:
# Vérification de l'état initial : statut catégoriel dans les métadonnées
conn_check = DuckLakeConnector(CATALOG_PATH, DATA_PATH, read_only=True).connect()

print("=== État initial — Métadonnées ===")
display(
    conn_check.execute(
        "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical"
        " DESC, name"
    ).fetchdf()
)

print(
    f"Lignes dans fact_table :"
    f" {conn_check.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}"
)
conn_check.close()

## 2 - Mise à jour de la base de données (`DatabaseUpdater`)

Cinq scénarios sont illustrés :

1. **Scénario 2.1** : mise à jour neutre (valeurs numériques seulement) — aucun changement de statut catégoriel
2. **Scénario 2.2** : l'upsert réduit les modalités de `source` à 5 valeurs (≤ seuil=10) → `is_categorical` **reste** `False`, puis correction manuelle via `update_column_metadata`
3. **Scénario 2.3** : l'upsert introduit 12 valeurs pour `model` (> seuil=10) → `is_categorical` **reste** `True`
4. **Scénario 2.4** : même mise à jour que 2.1, mais avec un DataFrame **polars** — compatibilité multi-backend narwhals
5. **Scénario 2.5** : colonne inconnue du DataFrame (§4.2) — refusée par défaut, acceptée explicitement via `allow_new_columns`

In [ ]:
# Création de la connexion DuckLake et initialisation de l'updater
# La connexion est partagée avec le DatabaseDeleter en Section 3
conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()
updater = DatabaseUpdater(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    enable_validation=True,
)

print(
    f"Lignes dans fact_table :"
    f" {conn.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}"
)

### Scénario 2.1 — Mise à jour sans changement de statut catégoriel

Mise à jour de quelques lignes existantes en modifiant uniquement les colonnes numériques (`value`, `quality_score`, `lower_bound`, `upper_bound`) et textuelles non-catégorielles (`notes`).

Aucune colonne catégorielle n'est modifiée, donc `is_categorical` reste inchangé pour toutes les colonnes.

In [ ]:
# Échantillon de 3 lignes existantes (les colonnes catégorielles portent déjà
# leurs libellés directement dans fact_table)
sample_21 = conn.execute("SELECT * FROM fact_table LIMIT 3").fetchdf()

# Modification des colonnes numériques uniquement
update_21 = sample_21.copy()
update_21["value"] = [99.0, 88.0, 77.0]
update_21["quality_score"] = [0.95, 0.85, 0.75]
update_21["lower_bound"] = [5.0, 4.0, 3.0]
update_21["upper_bound"] = [199.0, 180.0, 155.0]
update_21["notes"] = ["OK", "OK", "Warning"]

print("DataFrame de mise à jour (scénario 2.1) :")
display(update_21)

In [ ]:
# Vérification de l'état AVANT la mise à jour 2.1
print("=== AVANT la mise à jour 2.1 — Métadonnées ===")
display(
    conn.execute(
        "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical"
        " DESC, name"
    ).fetchdf()
)

In [ ]:
# Exécution de la mise à jour 2.1
success_21 = updater.update_database(
    update_df=update_21,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep="last",
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.1 : {'succès' if success_21 else 'échec'}")

In [ ]:
# Vérification de l'état APRÈS la mise à jour 2.1
# Résultat attendu : métadonnées inchangées (is_categorical identique à l'état AVANT)
print("=== APRÈS la mise à jour 2.1 — Métadonnées ===")
display(
    conn.execute(
        "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical"
        " DESC, name"
    ).fetchdf()
)

### Scénario 2.2 — Upsert réduisant les modalités de `source`

L'intégralité des lignes de `fact_table` est mise à jour avec 5 nouvelles valeurs pour `source` au lieu de 14. Bien que 5 ≤ seuil=10, `metadata.source.is_categorical` reste `False` : le statut est inféré une seule fois, à la création de la colonne, et n'est jamais recalculé par une écriture. Si le producteur veut un menu select pour `source`, il le déclare explicitement avec `update_column_metadata('source', is_categorical=True)` — un simple `UPDATE metadata`.

In [ ]:
# Reconstruction de TOUTES les lignes de la fact table
all_rows_22 = conn.execute("SELECT * FROM fact_table").fetchdf()

# Remplacement de source par 5 nouvelles valeurs (couverture uniforme sur toutes les
# lignes)
sources_new = ["alpha", "beta", "gamma", "delta", "epsilon"]
all_rows_22["source"] = [
    sources_new[i % len(sources_new)] for i in range(len(all_rows_22))
]

print(f"Lignes dans update_df          : {len(all_rows_22)}")
print(f"Valeurs uniques de source      : {sorted(all_rows_22['source'].unique())}")
print(
    f"Nombre de valeurs uniques      : {all_rows_22['source'].nunique()} (≤"
    f" seuil={CATEGORICAL_THRESHOLD}, statut inchangé)"
)

In [ ]:
# Vérification de l'état AVANT la mise à jour 2.2
print("Statut de source dans les métadonnées AVANT :")
display(
    conn.execute(
        "SELECT name, label, is_categorical FROM metadata WHERE name = 'source'"
    ).fetchdf()
)

In [ ]:
# Exécution de la mise à jour 2.2
success_22 = updater.update_database(
    update_df=all_rows_22,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep="last",
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.2 : {'succès' if success_22 else 'échec'}")

In [ ]:
# Vérification de l'état APRÈS la mise à jour 2.2
# Résultat attendu : source.is_categorical = False (jamais recalculé)
print("Statut de source dans les métadonnées APRÈS :")
display(
    conn.execute(
        "SELECT name, label, is_categorical FROM metadata WHERE name = 'source'"
    ).fetchdf()
)

print("\nValeurs distinctes de source dans fact_table (libellés d'origine) :")
display(
    conn.execute("SELECT DISTINCT source FROM fact_table ORDER BY source").fetchdf()
)

# Correction manuelle du statut catégoriel par le producteur
updater.update_column_metadata("source", is_categorical=True)
print("\nStatut de source après update_column_metadata(is_categorical=True) :")
display(
    conn.execute(
        "SELECT name, label, is_categorical FROM metadata WHERE name = 'source'"
    ).fetchdf()
)

### Scénario 2.3 — Upsert introduisant de nombreuses modalités pour `model`

Insertion de 12 nouvelles lignes dont la colonne `model` contient 12 valeurs uniques (model_A à model_L). Combinées aux 3 valeurs déjà présentes, `model` compte désormais 12 valeurs distinctes > seuil=10, mais `metadata.model.is_categorical` reste `True` : aucune écriture ne recalcule le statut.

> **Note** : après le scénario 2.2, `source` porte les valeurs `{alpha, beta, gamma, delta, epsilon}`. Les nouvelles lignes doivent utiliser une valeur existante pour `source`.

In [ ]:
# Construction de 12 nouvelles lignes, une par valeur de model (model_A à model_L)
# Combinaison de clé primaire fixe pour éviter tout conflit avec les lignes existantes
all_models = [f"model_{chr(65 + i)}" for i in range(12)]  # model_A ... model_L

rows_23 = []
for model in all_models:
    rows_23.append(
        {
            "indicator": "temperature",
            "country": "France",
            "kind": "forecast",
            "model": model,
            "training": "train_v1",
            "week": 52,
            "horizon": 30,
            "date": pd.Timestamp("2025-01-01"),
            "source": "alpha",  # valeur déjà présente dans fact_table.source
            "value": np.random.uniform(10, 100),
            "lower_bound": None,
            "upper_bound": None,
            "quality_score": np.random.uniform(0, 1),
            "notes": "OK",
        }
    )

update_23 = pd.DataFrame(rows_23)

print(f"Lignes dans update_df          : {len(update_23)}")
print(f"Valeurs uniques de model       : {sorted(update_23['model'].unique())}")

In [ ]:
# Exécution de la mise à jour 2.3
success_23 = updater.update_database(
    update_df=update_23,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep="last",
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.3 : {'succès' if success_23 else 'échec'}")

In [ ]:
# Vérification de l'état APRÈS la mise à jour 2.3
# Résultat attendu : model.is_categorical = True (inchangé malgré 12 valeurs)
print("Statut de model dans les métadonnées APRÈS :")
display(
    conn.execute(
        "SELECT name, label, is_categorical FROM metadata WHERE name = 'model'"
    ).fetchdf()
)

n_model = conn.execute("SELECT COUNT(DISTINCT model) FROM fact_table").fetchone()[0]
print(f"\nValeurs uniques de model dans fact_table : {n_model}")

### Scénario 2.4 — Compatibilité multi-backend narwhals (DataFrame polars)

`update_database()` est typé `IntoDataFrame` et convertit l'entrée en narwhals dès le point d'entrée.
Il accepte donc indifféremment un DataFrame **pandas** ou **polars** (ou tout autre backend narwhals-compatible).

Cet exemple reproduit le scénario 2.1 avec un DataFrame **polars**.

In [ ]:
# Conversion du DataFrame pandas update_21 en polars
update_24_polars = pl.from_pandas(update_21)
print(f"Type de l'entrée : {type(update_24_polars).__name__}")
display(update_24_polars)

In [ ]:
# Exécution de la mise à jour 2.4 avec un DataFrame polars
success_24 = updater.update_database(
    update_df=update_24_polars,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep="last",
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.4 (polars backend) : {'succès' if success_24 else 'échec'}")

### Scénario 2.5 — Colonnes inconnues (`allow_new_columns`, §4.2)

Une colonne du DataFrame absente de `fact_table` est **refusée par défaut** : `update_database` lève une `ValueError` plutôt que de l'ajouter silencieusement. Avec `allow_new_columns=True`, la colonne est ajoutée (type SQL inféré via `map_python_to_sql_type`), sa ligne `metadata` créée (`is_primary_key=False`, `is_categorical` inféré), et ses champs d'UI renseignés depuis `column_metadata`.

In [ ]:
# DataFrame portant une colonne inconnue de la fact table ('new_metric')
row_25 = {k: [sample_21[k].iloc[0]] for k in PRIMARY_KEYS}
update_25 = pd.DataFrame({**row_25, "new_metric": [1.0]})

# Sans allow_new_columns : ValueError
try:
    updater.update_database(
        update_df=update_25,
        check_duplicates_db=False,
        check_duplicates_update=False,
        keep="last",
        use_transaction=False,
    )
except ValueError as e:
    print(f"ValueError (attendue) : {e}")

In [ ]:
# Avec allow_new_columns=True : ajout explicite de la colonne et de ses métadonnées
success_25 = updater.update_database(
    update_df=update_25,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep="last",
    use_transaction=False,
    allow_new_columns=True,
    column_metadata={"new_metric": {"unit": "pts", "default_aggregation": "avg"}},
)
status_25 = "succès" if success_25 else "échec"
print(f"Mise à jour 2.5 (allow_new_columns=True) : {status_25}")

print("\nMétadonnées de new_metric :")
display(
    conn.execute(
        "SELECT name, sql_type, is_primary_key, unit, default_aggregation"
        " FROM metadata WHERE name = 'new_metric'"
    ).fetchdf()
)

## 3 - Suppression de données (`DatabaseDeleter`)

Deux scénarios sont illustrés :

1. **Scénario 3.1** : suppression des lignes de la Belgique (`model` conserve ses 12 valeurs, introduites en 2.3 avec `country='France'`)
2. **Scénario 3.2** : suppression des lignes `model_D` à `model_L` — `model` retrouve 3 valeurs uniques ; `is_categorical`, jamais recalculé, reste `True`

In [ ]:
# Initialisation du deleter en partageant la connexion ouverte par l'updater
# Les deux classes opèrent sur le même état de la base de données
deleter = DatabaseDeleter(
    connection=conn,
    enable_validation=True,
    auto_cleanup=True,
)

print(
    f"Lignes dans fact_table avant suppressions :"
    f" {conn.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}"
)

### Scénario 3.1 — Suppression filtrée sur un libellé

Suppression de toutes les lignes correspondant à la Belgique. `country` est catégorielle et stocke directement ses libellés dans `fact_table` — le filtre utilise donc la valeur `'Belgium'` telle quelle, sans jointure ni résolution d'ID.

In [ ]:
# Comptage préalable des lignes à supprimer
n_belgium = conn.execute(
    "SELECT COUNT(*) FROM fact_table WHERE country = 'Belgium'"
).fetchone()[0]
print(f"Lignes à supprimer : {n_belgium}")

In [ ]:
# Exécution de la suppression 3.1
deleted_31 = deleter.delete_rows(
    filters=[("country", "=", "Belgium")],
    use_transaction=True,
    perform_cleanup=True,
)
print(f"Lignes supprimées : {deleted_31}")

In [ ]:
# Vérification de l'état APRÈS la suppression 3.1
# Résultat attendu : model.is_categorical toujours True (statut jamais recalculé)
print("Statut de model dans les métadonnées :")
display(
    conn.execute(
        "SELECT name, label, is_categorical FROM metadata WHERE name = 'model'"
    ).fetchdf()
)

### Scénario 3.2 — Suppression réduisant les modalités de `model`

Suppression des lignes dont le modèle appartient à `{model_D, ..., model_L}`. Après la suppression, seuls `model_A`, `model_B`, `model_C` subsistent (3 valeurs ≤ seuil=10) : le nettoyage post-suppression (`perform_cleanup=True`) supprime les colonnes devenues entièrement nulles mais ne touche pas au statut catégoriel : `metadata.model.is_categorical` reste `True`.

In [ ]:
# Modèles à supprimer (ceux introduits en 2.3 qui dépassent le seuil)
models_to_delete = [f"model_{chr(65 + i)}" for i in range(3, 12)]  # model_D ... model_L
models_sql = ", ".join([f"'{m}'" for m in models_to_delete])
filter_32 = f"model IN ({models_sql})"

n_to_delete = conn.execute(
    f"SELECT COUNT(*) FROM fact_table WHERE model IN ({models_sql})"
).fetchone()[0]
print(f"Lignes à supprimer : {n_to_delete}")

In [ ]:
# Exécution de la suppression 3.2
deleted_32 = deleter.delete_rows(
    filters=filter_32,
    use_transaction=True,
    perform_cleanup=True,
)
print(f"Lignes supprimées : {deleted_32}")

In [ ]:
# Vérification de l'état APRÈS la suppression 3.2
# Résultat attendu : model.is_categorical = True (inchangé)
print("Statut de model dans les métadonnées :")
display(
    conn.execute(
        "SELECT name, label, is_categorical FROM metadata WHERE name = 'model'"
    ).fetchdf()
)

print("\nValeurs distinctes de model dans fact_table :")
display(conn.execute("SELECT DISTINCT model FROM fact_table ORDER BY model").fetchdf())

## 4 - Gestion explicite des colonnes (`add_columns` / `delete_columns`, §4.3)

Trois scénarios sont illustrés :

1. **Scénario 4.1** : ajout d'une colonne de valeurs à partir d'un DataFrame keyé sur les clés primaires
2. **Scénario 4.2** : refus d'une colonne déjà existante, puis remplacement explicite via `overwrite=True`
3. **Scénario 4.3** : une combinaison de clés du DataFrame absente de la base est **insérée** (fusion externe sur les clés primaires), les autres colonnes de valeur à `NULL`
4. **Scénario 4.4** : suppression de la colonne (`delete_columns`), une opération de métadonnées DuckLake — aucun fichier n'est réécrit
5. **Scénario 4.5** : recette de diffusion explicite (`get_key_combinations`) — un DataFrame porté par une clé *différente* de celle de la fact table n'est **pas** automatiquement diffusé ; c'est à l'appelant de le faire, explicitement, avant `add_columns`

### Scénario 4.1 — Ajout d'une colonne de valeurs

`add_columns` exige que le DataFrame porte **toutes** les clés primaires ; les autres colonnes sont les colonnes à ajouter. En interne : un seul `ALTER TABLE ... ADD COLUMN` par colonne nouvelle, puis un seul `UPDATE ... FROM` pour toutes les colonnes et un seul `INSERT ... SELECT` pour les combinaisons de clés absentes de la base, dans une unique transaction — en cas d'échec, ni la colonne, ni sa ligne `metadata`, ni les lignes insérées ne subsistent.

In [ ]:
# DataFrame keyé sur les clés primaires, portant une seule colonne de valeurs
key_sample = conn.execute(
    f"SELECT {', '.join(PRIMARY_KEYS)} FROM fact_table LIMIT 5"
).fetchdf()
score_df = key_sample.copy()
score_df["confidence_score"] = [0.9, 0.8, 0.95, 0.7, 0.85]

result_41 = updater.add_columns(
    score_df,
    column_metadata={"confidence_score": {"unit": "%", "default_aggregation": "avg"}},
)
print(f"add_columns (4.1) : {'succès' if result_41 else 'échec'}")

print("\nMétadonnées de confidence_score :")
display(
    conn.execute(
        "SELECT name, sql_type, is_primary_key, unit"
        " FROM metadata WHERE name = 'confidence_score'"
    ).fetchdf()
)

n_non_null = conn.execute(
    "SELECT COUNT(*) FROM fact_table WHERE confidence_score IS NOT NULL"
).fetchone()[0]
print(f"\nLignes non-NULL : {n_non_null} (attendu : {len(score_df)})")

### Scénario 4.2 — Colonne déjà existante : refus puis `overwrite=True`

In [ ]:
# Sans overwrite : refus (la colonne existe déjà depuis 4.1)
score_df2 = key_sample.iloc[:2].copy()
score_df2["confidence_score"] = [0.5, 0.4]

try:
    updater.add_columns(score_df2)
except ValueError as e:
    print(f"ValueError (attendue) : {e}")

In [ ]:
# Avec overwrite=True : les valeurs existantes sont remplacées
result_42 = updater.add_columns(score_df2, overwrite=True)
print(f"add_columns (4.2, overwrite=True) : {'succès' if result_42 else 'échec'}")

display(
    conn.execute(
        f"SELECT {', '.join(PRIMARY_KEYS)}, confidence_score FROM fact_table"
        f" WHERE {PRIMARY_KEYS[0]} = '{score_df2[PRIMARY_KEYS[0]].iloc[0]}'"
        f" AND {PRIMARY_KEYS[-1]} = '{score_df2[PRIMARY_KEYS[-1]].iloc[0]}'"
    ).fetchdf()
)

### Scénario 4.3 — Combinaison de clés absente de la base

Une ligne du DataFrame dont la combinaison de clés primaires n'existe pas dans `fact_table` est **insérée** : `add_columns` fusionne sur les clés primaires. La nouvelle ligne porte les clés et les colonnes du DataFrame ; les autres colonnes de valeur (`value`, `source`, …) valent `NULL`. Le rapport distingue `rows_updated` et `rows_inserted`.

In [ ]:
# Ligne portant une combinaison de clés qui n'existe pas dans fact_table
score_df3 = key_sample.iloc[:1].copy()
for col in PRIMARY_KEYS:
    if col == "date":
        score_df3[col] = pd.Timestamp("1999-01-01")
    elif col in ("week", "horizon"):
        score_df3[col] = 9999
    else:
        score_df3[col] = "does_not_exist"
score_df3["confidence_score"] = [0.1]

n_before = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
result_43 = updater.add_columns(score_df3, overwrite=True)
n_after = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]

print(result_43.summary())
print(f"Lignes avant : {n_before}, après : {n_after}")
print(f"rows_updated={result_43.rows_updated}, rows_inserted={result_43.rows_inserted}")
display(
    conn.execute(
        "SELECT indicator, value, source, confidence_score FROM fact_table"
        " WHERE indicator = 'does_not_exist'"
    ).fetchdf()
)

### Scénario 4.4 — Suppression de la colonne (`delete_columns`)

`ALTER TABLE ... DROP COLUMN` est, chez DuckLake, une opération de métadonnées : aucun fichier Parquet n'est réécrit (mesuré, §4.3/§5.4 de `specification-bdd.md`).

In [ ]:
# Suppression de confidence_score
result_44 = deleter.delete_columns(["confidence_score"], use_transaction=False)
print(f"delete_columns (4.4) : {result_44}")

columns_after = [row[0] for row in conn.execute("DESCRIBE fact_table").fetchall()]
print(f"confidence_score toujours présente : {'confidence_score' in columns_after}")

### Scénario 4.5 — Recette de diffusion explicite (`get_key_combinations`)

Une valeur connue à la seule granularité `model` (ex. un score par modèle) n'est **pas** diffusée automatiquement sur la clé complète de `fact_table` : la diffusion (broadcast) sur un sous-ensemble de clés n'est pas implémentée (§4.3 — un DataFrame porté par une autre clé est un autre jeu de résultats). Si l'utilisateur veut réellement diffuser la valeur, il joint explicitement son DataFrame partiel aux combinaisons de clés existantes (`get_key_combinations`) avant d'appeler `add_columns`.

In [ ]:
# Valeur connue à la seule granularité 'model'
model_bonus_df = pl.DataFrame(
    {"model": ["model_A", "model_B", "model_C"], "model_bonus": [10, 20, 30]}
)

# Combinaisons de clés existantes (toutes les clés primaires par défaut)
keys_45 = updater.get_key_combinations()
print(f"Colonnes de get_key_combinations() : {keys_45.columns}")

# Diffusion explicite : jointure sur 'model' avant add_columns
# (get_key_combinations renvoie un DataFrame narwhals à backend pyarrow : conversion
# vers polars avant la jointure)
broadcast_df = keys_45.to_polars().join(model_bonus_df, on="model", how="inner")
result_45 = updater.add_columns(broadcast_df)
print(f"add_columns (4.5, diffusion explicite) : {'succès' if result_45 else 'échec'}")

print("\nÉchantillon après diffusion :")
display(
    conn.execute(
        "SELECT DISTINCT model, model_bonus FROM fact_table"
        " WHERE model IN ('model_A', 'model_B', 'model_C') ORDER BY model"
    ).fetchdf()
)

conn.close()